# argmax-accuracy-eval — worked example 1: Per-class top-1 accuracy from logits

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `argmax-accuracy-eval`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Top-1 accuracy compares `logits.argmax(dim=-1)` against the labels. To get **per-class** accuracy you can't just take a global mean: you must compute, for each class `c`, the mean of `correct` restricted to the rows whose true label is `c`. A boolean mask `labels == c` selects those rows, and `correct[mask].float().mean()` is that class's accuracy.

## Worked solution

**Step 1 — predictions.** `preds = logits.argmax(dim=-1)` reduces the `(B, C)` logits to a `(B,)` vector of predicted class indices. Using `dim=-1` always points at the class axis regardless of leading dims.

**Step 2 — correctness vector.** `correct = (preds == labels)` is a `(B,)` boolean tensor, True where the prediction matched.

**Step 3 — loop over classes.** For each class `c` in `range(C)`, build the mask `mask = (labels == c)`. This is True only on rows whose ground-truth label is `c`.

**Step 4 — conditional mean.** `correct[mask]` keeps only the entries for that class; `.float().mean()` turns the booleans into 0.0/1.0 and averages them. That is exactly the fraction of class-`c` examples predicted correctly. We guard against an empty class (no examples) by emitting `nan` to avoid a divide-by-zero — here every class is present.

**Why it works.** Global accuracy weights every example equally; per-class accuracy re-weights so each class contributes its own recall, which is what you want when classes are imbalanced.

In [ ]:
def per_class_accuracy(logits, labels, num_classes):
    preds = logits.argmax(dim=-1)
    correct = (preds == labels)
    accs = []
    for c in range(num_classes):
        mask = (labels == c)
        if mask.sum().item() == 0:
            accs.append(float('nan'))
        else:
            accs.append(correct[mask].float().mean().item())
    return accs

t.manual_seed(0)
C = 3
logits = t.randn(12, C)
labels = t.randint(0, C, (12,))
print(per_class_accuracy(logits, labels, C))